In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    BooleanType, LongType, MapType
)
from pyspark.sql.functions import window, col
from datetime import datetime

In [0]:
# project data catalog defined and created in <placeholder> notebook
catalog = 'wikimedia_db'

# db_schema containing unprocessed/streaming data
uc_schema_raw_events = 'raw_events'

# raw data is saved in a temp volume by yy_mm_day
raw_events_volume_time = datetime.now()
raw_events_volume =  f"events_tmp_{raw_events_volume_time.strftime('%y_%m_%d')}"
raw_data_path = f'/Volumes/{catalog}/{uc_schema_raw_events}/{raw_events_volume}'

# db schema for checkpointing streaming tables
db_schema_checkpoints = 'checkpoints'
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db_schema_checkpoints}")

In [0]:

# Meta schema (nested)
meta_schema = StructType([
    StructField("uri", StringType(), True),
    StructField("request_id", StringType(), True),
    StructField("id", StringType(), True),
    StructField("dt", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("stream", StringType(), True)
])

# Length schema (nested)
length_schema = StructType([
    StructField("old", IntegerType(), True),
    StructField("new", IntegerType(), True)
])

# Revision schema (nested)
revision_schema = StructType([
    StructField("old", LongType(), True),
    StructField("new", LongType(), True)
])

# Main recent change schema
recentchange_schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("meta", meta_schema, True),
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("length", length_schema, True),
    StructField("revision", revision_schema, True),
    StructField("server_url", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("parsedcomment", StringType(), True),
])


In [0]:
# Read data from a file
# Similar to definition of staticInputDF above, just using `readStream` instead of `read`
streamingInputDF = (
  spark
    .readStream                       
    .schema(recentchange_schema)               # Set the schema of the JSON data
    .option("maxFilesPerTrigger", 1)  # Treat a sequence of files as a stream by picking n number of files at a time
    .json(raw_data_path)
)


In [0]:
# Do some transformations
# Same query as staticInputDF
streamingCountsDF = (
  streamingInputDF
    .groupBy(
      streamingInputDF.bot, # group by edit made by bot boolean
      window(
        col("timestamp").cast("timestamp"), 
        "5 minutes"
      )
    )
    .count()
)


In [0]:
# temp volume for checkpoint storage
volume = 'tmp_streamingInputDF'
volume_path = f'/Volumes/{catalog}/{db_schema_checkpoints}/{volume}'
volume_name = f'{catalog}.{db_schema_checkpoints}.{volume}'

# drop old temp volume and recreate
spark.sql(f"DROP VOLUME IF EXISTS {volume_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

# Display the streaming dataframe
streamingInputDF.display(checkpointLocation=volume_path)

In [0]:
# temp volume for checkpoint storage
volume = 'tmp_streamingDF'
volume_path = f'/Volumes/{catalog}/{db_schema_checkpoints}/{volume}'
volume_name = f'{catalog}.{db_schema_checkpoints}.{volume}'

# drop old temp volume and recreate
spark.sql(f"DROP VOLUME IF EXISTS {volume_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

# Display transformed data
streamingCountsDF.display(checkpointLocation=volume_path)

In [0]:
# BRONZE LAYER
print(" BRONZE LAYER - Creating schema and basic processing")
# Create Bronze schema
uc_schema_bronze = 'bronze_events'
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{uc_schema_bronze}")

# Transform raw data into clean format
from pyspark.sql.functions import col, when

bronze_df = streamingInputDF.select(
    col("id").alias("event_id"),
    col("title").alias("page_title"),
    col("user").alias("username"),
    col("timestamp").cast("timestamp").alias("event_time"),
    col("type").alias("change_type"),
    col("namespace"),
    col("bot").alias("is_bot"),
    col("minor").alias("is_minor"),
    col("server_name"),
    col("wiki"),
    col("comment"),
    col("length.old").alias("length_old"),
    col("length.new").alias("length_new"),
    (col("length.new") - col("length.old")).alias("size_change"),
    col("revision.new").alias("revision_id")
).filter(
    col("event_id").isNotNull() & col("page_title").isNotNull()
)

print(" Bronze transformation defined")

# Save to Bronze table with explicit checkpoint
bronze_table = f"{catalog}.{uc_schema_bronze}.wikimedia_changes"
bronze_checkpoint_vol = 'bronze_cp'

# Drop old checkpoint and create new one
spark.sql(f"DROP VOLUME IF EXISTS {catalog}.{db_schema_checkpoints}.{bronze_checkpoint_vol}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{db_schema_checkpoints}.{bronze_checkpoint_vol}")
bronze_checkpoint = f"/Volumes/{catalog}/{db_schema_checkpoints}/{bronze_checkpoint_vol}"

print(f" Writing to Bronze table: {bronze_table}")
print(f" Checkpoint location: {bronze_checkpoint}")


bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(bronze_table)
)

print(" Bronze stream started (processing all available data...)")

# Wait for completion
bronze_query.awaitTermination()
print(" Bronze processing completed!")


# Verify Bronze data (NO STREAMING - just read the table)
bronze_count = spark.read.table(bronze_table).count()
print(f" BRONZE: {bronze_count} events processed")

# Show sample (static read, not streaming)
print("\n Sample Bronze data:")
spark.read.table(bronze_table).show(5, truncate=False)

print("\n Distribution by change type:")
spark.read.table(bronze_table).groupBy("change_type").count().show()

In [0]:
# Silver LAYER

print(" SILVER LAYER - Time window aggregations")

# Read from Bronze (streaming)
bronze_stream = spark.readStream.table(bronze_table)

# Silver aggregations
from pyspark.sql.functions import window, count, avg, sum as spark_sum, abs as spark_abs

silver_df = (
    bronze_stream
    .withWatermark("event_time", "15 minutes")
    .groupBy(
        window("event_time", "10 minutes"),
        "server_name",
        "is_bot",
        "change_type"
    )
    .agg(
        count("*").alias("edit_count"),
        avg("size_change").alias("avg_size_change"),
        spark_sum(when(col("size_change") > 0, col("size_change")).otherwise(0)).alias("total_additions"),
        spark_sum(when(col("size_change") < 0, spark_abs(col("size_change"))).otherwise(0)).alias("total_deletions"),
        count(when(col("is_minor") == True, 1)).alias("minor_edits"),
        count(when(col("namespace") == 0, 1)).alias("main_namespace_edits")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "*"
    )
)

print(" Silver transformation defined")

# Save to Silver table with explicit checkpoint
silver_table = f"{catalog}.silver_metrics.edit_statistics"
silver_checkpoint_vol = 'silver_cp'

# Create Silver schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver_metrics")
spark.sql(f"DROP VOLUME IF EXISTS {catalog}.{db_schema_checkpoints}.{silver_checkpoint_vol}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{db_schema_checkpoints}.{silver_checkpoint_vol}")
silver_checkpoint = f"/Volumes/{catalog}/{db_schema_checkpoints}/{silver_checkpoint_vol}"

print(f" Writing to Silver table: {silver_table}")
print(f" Checkpoint location: {silver_checkpoint}")


silver_query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("complete")  
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .table(silver_table)
)

print(" Silver stream started")
silver_query.awaitTermination()
print(" Silver processing completed!")

# Verify Silver data (static read)
try:
    silver_count = spark.read.table(silver_table).count()
    print(f" SILVER: {silver_count} windows processed")
    
    if silver_count > 0:
        print("\n Sample Silver data:")
        spark.read.table(silver_table).orderBy("window_start", ascending=False).show(5)
    else:
        print(" No Silver data (not enough time variation in collected data)")
        print(" This is normal if all events were collected in < 10 minutes")
except Exception as e:
    print(f" Silver verification error: {e}")



In [0]:
#GOLD LAYER


print(" GOLD LAYER - Dashboard metrics")
# Read from Bronze (streaming)
bronze_stream_gold = spark.readStream.table(bronze_table)

# Gold metrics: hourly overview
gold_df = (
    bronze_stream_gold
    .withWatermark("event_time", "2 hours")
    .groupBy(
        window("event_time", "1 hour"),
        "server_name"
    )
    .agg(
        count("*").alias("total_events"),
        count(when(col("is_bot") == True, 1)).alias("bot_edits"),
        count(when(col("is_bot") == False, 1)).alias("human_edits"),
        count(when(col("change_type") == "new", 1)).alias("new_pages"),
        count(when(col("change_type") == "edit", 1)).alias("edits"),
        avg("size_change").alias("avg_change_size"),
        spark_sum("size_change").alias("net_content_change")
    )
    .select(
        col("window.start").alias("hour_start"),
        col("window.end").alias("hour_end"),
        "*"
    )
)

print(" Gold transformation defined")


# Save to Gold table with explicit checkpoint
gold_table = f"{catalog}.gold_dashboards.hourly_dashboard"
gold_checkpoint_vol = 'gold_cp'

# Create Gold schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold_dashboards")
spark.sql(f"DROP VOLUME IF EXISTS {catalog}.{db_schema_checkpoints}.{gold_checkpoint_vol}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{db_schema_checkpoints}.{gold_checkpoint_vol}")
gold_checkpoint = f"/Volumes/{catalog}/{db_schema_checkpoints}/{gold_checkpoint_vol}"

print(f" Writing to Gold table: {gold_table}")
print(f" Checkpoint location: {gold_checkpoint}")

#  Use availableNow with explicit checkpoint
gold_query = (
    gold_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", gold_checkpoint)
    .trigger(availableNow=True)
    .table(gold_table)
)

print(" Gold stream started")
gold_query.awaitTermination()
print(" Gold processing completed!")


# Verify Gold data (static read)
try:
    gold_count = spark.read.table(gold_table).count()
    print(f" GOLD: {gold_count} hourly windows processed")
    
    if gold_count > 0:
        print("\n Sample Gold data:")
        spark.read.table(gold_table).orderBy("hour_start", ascending=False).show(5)
    else:
        print(" No Gold data (need data spanning at least 1 hour)")
        print(" This is normal if all events were collected in < 60 minutes")
except Exception as e:
    print(f" Gold verification error: {e}")

In [0]:
#ALERT SYSTEM


print(" ALERT SYSTEM: Detecting rare events")

#Monitored rare events:
#1. Massive edits (>5000 characters added or removed)
#2. Large deletions (>3000 characters)
#3. Long new pages (>10000 characters)
#4. Edits to protected pages (special namespaces)


# Read from Bronze
bronze_stream_alerts = spark.readStream.table(bronze_table)

# Detect rare events
rare_events_df = bronze_stream_alerts.filter(
    # Condition 1: Massive edits
    (spark_abs(col("size_change")) > 1000) |
    # Condition 2: Very long new pages
    ((col("change_type") == "new") & (col("length_new") > 5000)) |
    # Condition 3: Edits in Wikipedia (4) or MediaWiki (8) namespace
    (col("namespace").isin([4, 8]))
).select(
    "*",
    when(spark_abs(col("size_change")) > 1000, "MASSIVE_EDIT")
    .when((col("change_type") == "new") & (col("length_new") > 5000), "LARGE_NEW_PAGE")
    .when(col("namespace").isin([4, 8]), "SPECIAL_NAMESPACE")
    .otherwise("OTHER").alias("alert_type"),
    col("event_time").alias("alert_time")
)

print( Alert filters defined")


# Save to Alerts table
alert_table = f"{catalog}.alerts.rare_events"
alert_checkpoint_vol = 'alert_cp'

# Create Alerts schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.alerts")
spark.sql(f"DROP VOLUME IF EXISTS {catalog}.{db_schema_checkpoints}.{alert_checkpoint_vol}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{db_schema_checkpoints}.{alert_checkpoint_vol}")
alert_checkpoint = f"/Volumes/{catalog}/{db_schema_checkpoints}/{alert_checkpoint_vol}"

print(f" Writing to Alerts table: {alert_table}")

alert_query = (
    rare_events_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", alert_checkpoint)
    .trigger(availableNow=True)
    .table(alert_table)
)

print(" Alert stream started")


# Visualize alerts in real-time
display(
    spark.readStream.table(alert_table)
    .select("alert_time", "alert_type", "page_title", "username", "size_change", "comment")
    .orderBy(col("alert_time").desc())
)


# Stop Alerts stream
alert_query.stop()
print(" Alert stream stopped")


# Statistics on detected alerts
print(" ALERT STATISTICS:")
alert_stats = spark.read.table(alert_table)

if alert_stats.count() > 0:
    alert_stats.groupBy("alert_type").count().orderBy("count", ascending=False).show()
    print(f"\n Total alerts: {alert_stats.count()}")
else:
    print(" No alerts detected yet (collect more data)")


In [0]:
#FINAL ANALYSIS

print(" ANALYSIS OF COLLECTED DATA")

# Bronze stats
bronze_df_static = spark.read.table(bronze_table)
print(f"\n BRONZE - Total events: {bronze_df_static.count()}")
bronze_df_static.groupBy("server_name", "change_type").count().orderBy("count", ascending=False).show()

# Silver stats
silver_df_static = spark.read.table(silver_table)
if silver_df_static.count() > 0:
    print(f"\n SILVER - Total windows: {silver_df_static.count()}")
    silver_df_static.orderBy("window_start", ascending=False).show(10)
else:
    print(" No Silver data (aggregations need more time)")


# Gold stats
gold_df_static = spark.read.table(gold_table)
if gold_df_static.count() > 0:
    print(f"\n GOLD - Total hours: {gold_df_static.count()}")
    gold_df_static.orderBy("hour_start", ascending=False).show(10)
else:
    print(" No Gold data (hourly aggregations need more time)")


# Top edited pages
print("\n TOP 10 MOST EDITED PAGES:")
bronze_df_static.groupBy("page_title", "server_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10, truncate=False)

# Bots vs Humans comparison
print("\n BOTS vs  HUMANS:")
bronze_df_static.groupBy("is_bot") \
    .agg(
        count("*").alias("total_edits"),
        avg("size_change").alias("avg_change_size")
    ) \
    .show()